<a href="https://colab.research.google.com/github/amiralito/Protenix_Colab/blob/main/protenix_v2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protenix-v2 on Colab — batch predictions

Run **many complexes in one pass** with [Protenix-v2](https://github.com/bytedance/Protenix) (ByteDance, AF3-class), using the **ColabFold MSA server** (`api.colabfold.com`) so **no local sequence databases** are needed. Unique sequences are MSA-searched **once** and reused across jobs. For a single structure with full 3D/pLDDT inspection, use the companion **single-prediction** notebook.

**Setup:** Runtime → Change runtime type → **GPU**. Use an **A100 (40 GB)** for large complexes / resistosomes (≈67 GB at 2000 tokens); a T4 is fine for batches of small dimers.

**Flow:**
1. Run **Setup** cells 1–3 once per session (install + helpers).
2. Set options in **Run settings** (cell 5) — model, seeds, samples.
3. **B1** loads your batch list (build it with `make_protenix_batch.py`).
4. *(Optional)* **B2-pre** enables dense PAE capture — run **before** B2.
5. **B2** runs MSA + inference over the whole batch.
6. **B3** summary · **B4** master confidence table · **B5** PAE export (ChimeraX/ipSAE npz) · **B6** ipSAE interface scores · **B7** per-job means across seeds · **B8** model viewer (MolView).
7. Zip & download / archive (**10**, **10b**).

Per-hit 3D/PAE inspection: open the model + `pae_<cifstem>.npz` in **ChimeraX**, or load that job in the single-prediction notebook.

## 1. GPU check

In [ ]:
#@title 1. GPU check { display-mode: "form" }
import subprocess
out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
if not out:
    raise SystemExit('STOP: No GPU. Runtime -> Change runtime type -> GPU.')
print(out)
name, mem = [x.strip() for x in out.split(',')]
mem_gb = int(mem.split()[0]) / 1024
print(f'Detected: {name}  (~{mem_gb:.0f} GB)')
if mem_gb < 15:
    print('WARNING: <15 GB VRAM. Only very small monomers will fit. Prefer A100.')
elif mem_gb < 24:
    print('OK for monomers / small complexes (<~1000 tokens). Large assemblies may OOM -> use A100.')
else:
    print('Plenty of VRAM for large complexes.')

## 2. Install Protenix + tools (run once per session)

In [ ]:
#@title 2. Install Protenix + tools (run once per session) { display-mode: "form" }
import os, sys, subprocess, importlib.util

def sh(cmd):
    print('+', cmd); subprocess.run(cmd, shell=True, check=True)
def have(m):
    return importlib.util.find_spec(m) is not None

# 1) Protenix + light viewers (only if missing, so re-running never disturbs a torch fix).
if not have('protenix'):
    sh('pip -q install --upgrade protenix --index-url https://pypi.org/simple')
else:
    print('protenix already installed - skipping its pip.')
for _p in ('py3Dmol', 'gemmi'):
    if not have(_p):
        sh(f'pip -q install {_p}')
sh('apt-get -qq install -y kalign hmmer >/dev/null 2>&1 || true')

# 2) Online ColabFold MSA server (no local databases).
os.environ['MMSEQS_SERVICE_HOST_URL'] = 'https://api.colabfold.com'

# 3) GPU compute capability via nvidia-smi - NO torch import here, on purpose.
cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip().splitlines()[0].strip()
os.environ['TORCH_CUDA_ARCH_LIST'] = cc          # '12.0' Blackwell, '8.0' A100, etc.

# 4) Blackwell (sm_120): stock/cu126 wheels lack its kernels, AND the pinned 2.7.1+cu128 we
#    used before covered core ops but was missing some sm_120 kernels (e.g. FlashAttention).
#    So we do NOT auto-install a torch here. On sm_120, install the nightly cu128 build via
#    cell 2c ONCE, restart the runtime, then re-run this cell. (No torch touched here = no overlap.)
if cc.replace('.', '') == '120':
    # Protenix's fused layernorm (fastfold_layer_norm_cuda) is compiled without sm_120 and is
    # gated by LAYERNORM_TYPE, NOT by --enable_fusion. Force native PyTorch layernorm on Blackwell.
    os.environ['LAYERNORM_TYPE'] = 'torch'
    # Auto-fix: Blackwell (sm_120) needs the nightly cu128 torch (full sm_120 kernels incl.
    # FlashAttention). We check in a SUBPROCESS (no torch import here, so no restart is needed)
    # and install it automatically only if the current torch lacks sm_120.
    _chk = subprocess.run([sys.executable, '-c',
        "import torch,sys; sys.exit(0 if 'sm_120' in ' '.join(torch.cuda.get_arch_list()) else 1)"],
        capture_output=True)
    if _chk.returncode == 0:
        print('** Blackwell (sm_120): torch already carries sm_120 kernels - good to go. **')
    else:
        print('** Blackwell (sm_120) detected: installing nightly cu128 torch automatically '
              '(~2-3 min, one-time). **')
        subprocess.run('pip -q install --pre --force-reinstall torch '
                       '--index-url https://download.pytorch.org/whl/nightly/cu128',
                       shell=True, check=False)
        print('   Done - LAYERNORM_TYPE=torch set, no restart needed. Continue to the next cells.')

print(f'Setup done | compute_cap {cc} | TORCH_CUDA_ARCH_LIST={cc}')
print('MMSEQS_SERVICE_HOST_URL =', os.environ['MMSEQS_SERVICE_HOST_URL'])
print('Note: torch is not imported in this cell, so the Blackwell auto-fix needs no restart.')

## 2c. Blackwell (sm_120) torch — optional fallback (only if the auto-fix in cell 2 fails)

Cell 2 now installs the nightly cu128 torch **automatically** on Blackwell cards (RTX PRO 6000 /
RTX 50-series, sm_120), with no restart needed. Run this cell only if that auto-fix fails or you
want to force a clean reinstall; it fetches the nightly cu128 build (full sm_120 kernel set incl.
FlashAttention). **After it finishes, restart the runtime** (Runtime -> Restart session, which
keeps installed packages + Drive), then re-run from cell 2. Non-Blackwell cards (T4/L4/A100) never
need this — stock torch already works.

In [ ]:
#@title 2c. Blackwell (sm_120) torch — optional fallback (only if cell 2 auto-fix fails) { display-mode: "form" }
import os, subprocess
os.environ['TORCH_CUDA_ARCH_LIST'] = '12.0'
# Nightly cu128 (with deps) - last resort if the pinned stable build ever lacks sm_120 coverage.
subprocess.run('pip install --pre --force-reinstall torch '
               '--index-url https://download.pytorch.org/whl/nightly/cu128',
               shell=True, check=True)
print('\nDONE. Now: Runtime -> Restart session, then re-run from cell 2.')
print('Verify after restart:  import torch; print(torch.cuda.get_arch_list())  -> must include sm_120')


## 3. (Recommended) Save outputs to Google Drive + set working directories

Mounting Drive lets predictions, the MSA cache and the weights persist across disconnects. This cell also creates the input / MSA / output directories, so run it **before** the helpers. Untick `use_drive` to use local `/content` storage instead.

In [ ]:
#@title 3. Google Drive + working directories { display-mode: "form" }
# Mount Drive (recommended) so predictions, the MSA cache and the weights survive a disconnect,
# then create the working dirs. Untick use_drive to keep everything in local /content.
use_drive = True  #@param {type:"boolean"}
custom_output_dir = ""  #@param {type:"string"}
import os, shutil, time

if use_drive:
    from google.colab import drive
    _mp = '/content/drive'
    if os.path.ismount(_mp):
        print('Drive already mounted.')
    else:
        if os.path.isdir(_mp) and os.listdir(_mp):
            _bak = f'{_mp}_old_{int(time.time())}'
            shutil.move(_mp, _bak)
            print(f'Moved stale {_mp} -> {_bak} (leftover files, not live Drive).')
        drive.mount(_mp)
    WORK = '/content/drive/MyDrive/protenix_run'
else:
    WORK = '/content/protenix_run'

INPUT_DIR = os.path.join(WORK, 'inputs')
MSA_DIR   = os.path.join(WORK, 'msa')
OUT_DIR   = custom_output_dir.strip() or os.path.join(WORK, 'outputs')
for d in (INPUT_DIR, MSA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print('Working dir:', WORK)
print('Outputs ->', OUT_DIR)

## 3b. Helpers (run once)

Input-JSON builders, the weights fetcher, the MSA cache and the shared analysis helpers every results cell uses. Run once, after cell 3.

In [ ]:
#@title 3b. Helpers (run once) { display-mode: "form" }
import os, json, glob, time

JOB_SPEC = None  # set by an input cell

def _clean_seq(s):
    return ''.join(s.split()).upper()

def protein(seq, count=1, ids=None):
    d = {'sequence': _clean_seq(seq), 'count': int(count)}
    if ids: d['id'] = ids
    return {'proteinChain': d}

def ligand(spec, count=1):
    # spec: 'CCD_ATP'  OR  a SMILES string  OR  'FILE_/path/to/lig.sdf'
    return {'ligand': {'ligand': spec, 'count': int(count)}}

def ion(code, count=1):
    # code is a bare CCD code WITHOUT the CCD_ prefix, e.g. 'MG', 'ZN', 'NA'
    return {'ion': {'ion': code, 'count': int(count)}}

def dna(seq, count=1):
    return {'dnaSequence': {'sequence': _clean_seq(seq), 'count': int(count)}}

def rna(seq, count=1):
    return {'rnaSequence': {'sequence': _clean_seq(seq), 'count': int(count)}}

def make_spec(name, entities):
    name = name.strip().replace(' ', '_') or 'job'
    return {'name': name, 'sequences': entities}

def write_json(spec, path=None):
    path = path or os.path.join(INPUT_DIR, spec['name'] + '.json')
    with open(path, 'w') as f:
        json.dump([spec], f, indent=2)   # top level must be a LIST
    print('Wrote', path)
    print(json.dumps([spec], indent=2)[:1200])
    return path

HF_V2_REPO = 'TMF001/protenix-v2-weights'
HF_V2_FILE = 'protenix-v2.pt'
HF_V2_URL  = f'https://huggingface.co/{HF_V2_REPO}/resolve/main/{HF_V2_FILE}?download=true'
V2_CKPT    = '/root/checkpoint/protenix-v2.pt'          # where `-n protenix-v2` loads from

def _hf_token():
    """HF token from the HF_TOKEN env var or a Colab secret named HF_TOKEN (key icon, left bar)."""
    t = os.environ.get('HF_TOKEN', '').strip()
    if not t:
        try:
            from google.colab import userdata
            t = (userdata.get('HF_TOKEN') or '').strip()
        except Exception:
            t = ''
    return t or None

def _ckpt_ok(p):
    return bool(p) and os.path.isfile(p) and os.path.getsize(p) > 1e9   # error pages are KB, not GB

def _fast_download(url, out):
    """huggingface_hub/hf_xet (parallel) -> aria2c (16 connections) -> curl. Returns (ok, reason)."""
    import subprocess, importlib.util, shutil
    tok, why = _hf_token(), []
    try:
        if importlib.util.find_spec('hf_xet') is None:          # parallel chunked transfer
            subprocess.run('pip -q install -U "huggingface_hub[hf_xet]"', shell=True, check=False)
        os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')
        import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=HF_V2_REPO, filename=HF_V2_FILE,
                            local_dir=os.path.dirname(out), token=tok)
        if os.path.abspath(p) != os.path.abspath(out):
            shutil.move(p, out)
        if _ckpt_ok(out):
            return True, ''
    except Exception as e:
        why.append(f'huggingface_hub: {type(e).__name__}: {str(e).splitlines()[0][:160]}')
    hdr = f'--header="Authorization: Bearer {tok}" ' if tok else ''
    if subprocess.run('which aria2c', shell=True, capture_output=True).returncode != 0:
        subprocess.run('apt-get -qq install -y aria2 >/dev/null 2>&1 || true', shell=True, check=False)
    if subprocess.run('which aria2c', shell=True, capture_output=True).returncode == 0:
        d, f = os.path.dirname(out), os.path.basename(out)
        subprocess.run(f'aria2c -x16 -s16 -k1M -c --console-log-level=error {hdr}-d "{d}" -o "{f}" "{url}"',
                       shell=True, capture_output=True)
        if _ckpt_ok(out):
            return True, ''
    h = f'-H "Authorization: Bearer {tok}" ' if tok else ''
    r = subprocess.run(f'curl -sS -fL --retry 3 {h}-o "{out}" -w "%{{http_code}}" "{url}"',
                       shell=True, capture_output=True, text=True)
    if _ckpt_ok(out):
        return True, ''
    why.append(f'curl: HTTP {r.stdout.strip() or "?"} {r.stderr.strip()[:120]}')
    try:
        os.remove(out)                                   # never leave a partial/error file behind
    except OSError:
        pass
    return False, ' | '.join(why)

def ensure_v2_weights(dest_dir=None, verbose=True):
    """Stage protenix-v2 weights for `-n protenix-v2`. Returns the checkpoint path, or None.
       Looks for (1) an already-staged checkpoint, (2) `dest_dir` (a folder holding
       protenix-v2.pt, or the .pt file itself), (3) WORK/weights/protenix-v2.pt; otherwise
       downloads once into the persistent folder. Never reports success without a real file."""
    import shutil
    os.makedirs(os.path.dirname(V2_CKPT), exist_ok=True)
    if _ckpt_ok(V2_CKPT):
        return V2_CKPT
    d = dest_dir.strip() if isinstance(dest_dir, str) and dest_dir.strip() else None
    if d and d.endswith('.pt'):
        keep, store = d, os.path.dirname(d)
    else:
        store = d or os.path.join(globals().get('WORK', '/content/protenix_run'), 'weights')
        keep = os.path.join(store, HF_V2_FILE)
    if _ckpt_ok(keep):
        if verbose: print(f'Found Protenix-v2 weights: {keep} ({os.path.getsize(keep)/1e9:.2f} GB) - staging.')
        shutil.copy(keep, V2_CKPT)
        return V2_CKPT
    os.makedirs(store, exist_ok=True)
    if verbose: print(f'Downloading Protenix-v2 weights (~1.9 GB, one-time) -> {keep} ...')
    ok, why = _fast_download(HF_V2_URL, keep)
    if ok:
        if verbose: print(f'  -> saved ({os.path.getsize(keep)/1e9:.2f} GB). Future runs reuse this file.')
        shutil.copy(keep, V2_CKPT)
        return V2_CKPT
    print('\n*** Protenix-v2 weights are NOT available. ***')
    print('   ', why)
    print('   The HuggingFace mirror refuses anonymous downloads (401) and the ByteDance CDN returns')
    print('   403 outside CN (bytedance/Protenix#295). Options:')
    print('   1) If you have access to the mirror, add your token as a Colab secret named HF_TOKEN')
    print('      (key icon in the left bar) and re-run.')
    print(f'   2) Put protenix-v2.pt in {store} yourself (or point weights_dir in 3c at it).')
    print('   3) Do nothing: inference falls back to protenix_base_20250630_v1.0.0.')
    return None

# ---------------- shared analysis helpers (used by every results cell) ----------------
import re as _re, numpy as _np

def _run_dir():
    """Run folder to analyse: RUN_DIR if set (this session's run, or one you point it at),
       else the most recent OUT_DIR/run_*. Never the bare OUT_DIR, which mixes runs + archives."""
    rd = globals().get('RUN_DIR')
    if rd and os.path.isdir(rd):
        return rd
    runs = sorted(glob.glob(os.path.join(OUT_DIR, 'run_*')), key=os.path.getmtime)
    if runs:
        print(f'RUN_DIR not set - analysing the most recent run: {runs[-1]}\n'
              f'(set RUN_DIR = "<folder>" to analyse another run or an _archive/<stamp> folder)')
        return runs[-1]
    return OUT_DIR

def _in_archive(path, root):
    return '_archive' in os.path.relpath(path, root).split(os.sep)

def _summary_jsons(root):
    """All summary_confidence JSONs under root, skipping _archive copies (unless root IS an archive)."""
    return sorted(p for p in glob.glob(os.path.join(root, '**', '*summary_confidence*.json'),
                                       recursive=True) if not _in_archive(p, root))

def _job_of(path, root):
    parts = os.path.relpath(path, root).split(os.sep)
    if parts and parts[0] == '_archive' and len(parts) > 2:
        parts = parts[2:]
    return parts[0] if parts and parts[0] != '..' else 'NA'

def _seed_of(path):
    """Seed from a seed_<n> folder, else from the file name (..._seed<n>.ext, as 10b archives
       name files), else from a purely numeric folder. 'NA' only if none of these exist."""
    parts = path.split(os.sep)
    for x in parts:
        m = _re.fullmatch(r'seed_?(\d+)', x)
        if m:
            return m.group(1)
    m = _re.search(r'(?:^|_)seed_?(\d+)(?=[_.])', os.path.basename(path))
    if m:
        return m.group(1)
    for x in parts[:-1]:
        if x.isdigit():
            return x
    return 'NA'

def _sample_of(path):
    m = _re.search(r'sample_(\d+)', os.path.basename(path))
    return m.group(1) if m else 'NA'

def _pair_iptm(d):
    """Chain-pair ipTM summary. With 2 chains the single off-diagonal value IS ipTM, so the
       extra columns are NaN; with >=3 chains (ligand/ion chains count) they give the strongest
       and weakest pair. Same definition in every cell."""
    try:
        a = _np.array(d.get('chain_pair_iptm'), dtype=float)
    except Exception:
        a = _np.array([])
    n = int(a.shape[0]) if a.ndim == 2 else 0
    if n >= 3:
        off = a[~_np.eye(n, dtype=bool)]
        return dict(n_chains=n, chain_pair_iptm_max=float(_np.nanmax(off)),
                    chain_pair_iptm_min=float(_np.nanmin(off)))
    return dict(n_chains=n, chain_pair_iptm_max=_np.nan, chain_pair_iptm_min=_np.nan)

def _warn_na_seeds(seeds, where):
    n = sum(1 for s in seeds if str(s) == 'NA')
    if n:
        print(f'WARNING ({where}): {n} model(s) have no detectable seed - per-seed statistics '
              f'will be wrong for them. Point RUN_DIR at a run folder or an _archive/<stamp> folder.')

# ---- ipSAE (Boltz mode) inputs: per-token pLDDT + chain-pair ipTM next to the PAE ----
_IPSAE_RES = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS","MET",
              "PHE","PRO","SER","THR","TRP","TYR","VAL","DA","DC","DT","DG","A","C","U","G"}

def _ipsae_tokens(cif):
    """Per-token pLDDT (CIF B-factors) in exactly ipsae.py's token order, plus polymer-chain
       order (ipsae's chain indices) and all-chain order (Protenix chain_pair_iptm order)."""
    fields, n, toks, poly, allc = {}, 0, [], [], []
    with open(cif) as fh:
        for line in fh:
            if line.startswith('_atom_site.'):
                fields[line.strip().split('.', 1)[1]] = n; n += 1; continue
            if not line.startswith(('ATOM', 'HETATM')):
                continue
            t = line.split()
            an, rn = t[fields['label_atom_id']], t[fields['label_comp_id']]
            ch = t[fields['auth_asym_id']] if 'auth_asym_id' in fields else t[fields['label_asym_id']]
            b = float(t[fields['B_iso_or_equiv']])
            if ch not in allc:
                allc.append(ch)
            if t[fields['label_seq_id']] == '.':          # ligand atom -> own token
                toks.append(b); continue
            if an == 'CA' or 'C1' in an:                  # residue token
                toks.append(b)
                if ch not in poly:
                    poly.append(ch)
            elif rn not in _IPSAE_RES:                    # modified residue -> per-atom tokens
                toks.append(b)
    return _np.asarray(toks, dtype=_np.float32), poly, allc

def write_ipsae_inputs(wd, cif, pae, summary_json=None):
    """Stage a Boltz-mode ipSAE workspace in wd (model.cif, pae_/plddt_/confidence_ files).
       Without plddt_*.npz ipsae.py uses pLDDT = 0, which pins pDockQ/pDockQ2 to 0.0183/0.0073.
       Returns (pae_path, cif_path, plddt_ok, iptm_ok)."""
    import shutil
    os.makedirs(wd, exist_ok=True)
    cif_dst = os.path.join(wd, 'model.cif'); shutil.copy(cif, cif_dst)
    pae = _np.asarray(pae, dtype=_np.float32)
    pae_p = os.path.join(wd, 'pae_model.npz'); _np.savez(pae_p, pae=pae)
    plddt, poly, allc = _ipsae_tokens(cif)
    plddt_ok = len(plddt) == pae.shape[0]
    if plddt_ok:
        _np.savez(os.path.join(wd, 'plddt_model.npz'), plddt=plddt)
    iptm_ok = False
    if summary_json and os.path.exists(summary_json):
        try:
            cpi = _np.asarray(json.load(open(summary_json)).get('chain_pair_iptm'), dtype=float)
            idx = None
            if cpi.ndim == 2 and cpi.shape[0] == len(allc):
                idx = {c: allc.index(c) for c in poly}
            elif cpi.ndim == 2 and cpi.shape[0] == len(poly):
                idx = {c: i for i, c in enumerate(poly)}
            if idx:
                pci = {str(i): {str(j): float(cpi[idx[a], idx[b]]) for j, b in enumerate(poly) if a != b}
                       for i, a in enumerate(poly)}
                json.dump({'pair_chains_iptm': pci}, open(os.path.join(wd, 'confidence_model.json'), 'w'))
                iptm_ok = True
        except Exception:
            pass
    return pae_p, cif_dst, plddt_ok, iptm_ok

# ---- MSA: one ColabFold search per UNIQUE sequence, cached and reused across jobs/runs ----
_MSA_DRIVER = r'''
import hashlib, json, logging, os, shutil, signal, sys, time
json_in, json_out, cache, host, tmo, retries = sys.argv[1:7]
tmo, retries = int(tmo), int(retries)
class _Quiet(logging.Filter):            # Protenix logs routine polling at ERROR level
    def filter(self, r):
        return 'Sleeping for' not in r.getMessage()
logging.getLogger('protenix.web_service.colab_request_utils').addFilter(_Quiet())
from protenix.web_service.colab_request_utils import run_mmseqs2_service
class _Timeout(BaseException):           # BaseException: not swallowed by the client's retry loops
    pass
def _alarm(*_):
    raise _Timeout()
signal.signal(signal.SIGALRM, _alarm)
data = json.load(open(json_in))
seqs = []
for job in data:
    for s in job.get('sequences', []):
        pc = s.get('proteinChain')
        if pc and not pc.get('unpairedMsaPath') and pc['sequence'] not in seqs:
            seqs.append(pc['sequence'])
n_chains = sum(1 for job in data for s in job.get('sequences', []) if 'proteinChain' in s)
print(f'MSA: {n_chains} protein chains across {len(data)} job(s) -> {len(seqs)} unique sequence(s)', flush=True)
done, reused, searched, failed, t0 = {}, 0, 0, [], time.time()
for k, seq in enumerate(seqs, 1):
    key = hashlib.sha1(seq.encode()).hexdigest()[:20]
    d = os.path.join(cache, key)
    a3m = os.path.join(d, '0', 'non_pairing.a3m')
    tag = f'  [{k}/{len(seqs)}] {len(seq)} aa ({key[:8]})'
    if os.path.isfile(a3m) and os.path.getsize(a3m) > 0:
        done[seq] = a3m; reused += 1; print(f'{tag}: cached', flush=True); continue
    ok, t = False, time.time()
    for att in range(retries + 1):
        shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
        signal.alarm(tmo)
        try:
            run_mmseqs2_service(f'>query_0\n{seq}', d, use_env=True, use_filter=True,
                                use_templates=False, filter=None, use_pairing=False,
                                pairing_strategy='greedy', host_url=host,
                                user_agent='colabfold/1.5.5', server_mode='colabfold')
            signal.alarm(0)
            ok = os.path.isfile(a3m) and os.path.getsize(a3m) > 0
            if ok:
                break
            print(f'{tag}: server returned no alignment (attempt {att + 1}/{retries + 1})', flush=True)
        except _Timeout:
            print(f'{tag}: no result after {tmo / 60:g} min (queued/running) - resubmitting '
                  f'(attempt {att + 1}/{retries + 1})', flush=True)
        except Exception as e:
            signal.alarm(0)
            print(f'{tag}: server error: {str(e)[:200]} (attempt {att + 1}/{retries + 1})', flush=True)
            time.sleep(30)
    if ok:
        done[seq] = a3m; searched += 1; print(f'{tag}: searched in {time.time() - t:.0f}s', flush=True)
    else:
        failed.append(seq); print(f'{tag}: FAILED', flush=True)
for job in data:
    for s in job.get('sequences', []):
        pc = s.get('proteinChain')
        if pc and pc.get('sequence') in done and not pc.get('unpairedMsaPath'):
            pc['unpairedMsaPath'] = done[pc['sequence']]
json.dump(data, open(json_out, 'w'), indent=2)
print(f'MSA done: {reused} cached, {searched} searched, {len(failed)} failed '
      f'({time.time() - t0:.0f}s)', flush=True)
sys.exit(1 if failed else 0)
'''

def msa_dedup(json_in, json_out, timeout_min=20, retries=2, host='https://api.colabfold.com'):
    """Search each UNIQUE protein sequence once on the ColabFold server, cache the alignment
       in MSA_DIR/_seqcache/<sha1> (persists on Drive), and write json_out with unpairedMsaPath
       set on every chain. A search still queued/running after timeout_min is resubmitted
       (up to `retries` times). Same alignments Protenix's own colabfold mode produces (its
       paired search is written to a folder it never links, so pairing is unchanged).
       Returns True only if every sequence got an MSA."""
    import subprocess, sys
    drv = '/content/_msa_dedup.py'
    with open(drv, 'w') as fh:
        fh.write(_MSA_DRIVER)
    cache = os.path.join(MSA_DIR, '_seqcache'); os.makedirs(cache, exist_ok=True)
    env = dict(os.environ, TQDM_DISABLE='1', MMSEQS_SERVICE_HOST_URL=host,
               PROTENIX_DUMP_PAE='0')      # keep the PAE startup patch out of this process
    p = subprocess.Popen([sys.executable, drv, json_in, json_out, cache, host,
                          str(int(timeout_min * 60)), str(int(retries))],
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    return p.wait() == 0

# ---- inference: shared command builder + streaming runner ----
def _stream(cmd, env=None, verbose=False):
    import subprocess
    print('\n+ ' + ' '.join(cmd), flush=True)
    e = dict(os.environ)
    if env: e.update(env)
    p = subprocess.Popen(cmd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    buf = []
    for line in p.stdout:
        buf.append(line)
        low = line.lower()
        if 'sleeping for' in low:
            continue
        if (verbose
                or any(k in low for k in ('error', 'traceback', 'failed', 'exception',
                                          'out of memory', 'warning', 'file "', '^^^'))
                or any(k in line for k in ('completed', 'Job completed', 'N_token', 'Downloading',
                                           'Saved', 'Seed', 'Predict', 'Inference', '[pae_dump]'))
                or '%|' in line or 'it/s' in line or 's/it' in line
                or 'INFO' not in line):
            print(line, end='')
    p.wait()
    return p.returncode, ''.join(buf)

V2_FALLBACK = 'protenix_base_20250630_v1.0.0'     # v1 architecture, newer training cutoff (2025-06-30)

def resolve_model():
    """protenix-v2 if its weights are found (Run-settings path, 3c path, Drive cache) or can be
       downloaded; otherwise switch CFG to V2_FALLBACK, loudly. Returns the model name used."""
    if CFG['model_name'] != 'protenix-v2':
        return CFG['model_name']
    path = (CFG.get('v2_weights_path') or '').strip() or (globals().get('weights_dir') or '').strip()
    if ensure_v2_weights(dest_dir=path or None):
        print('Model: protenix-v2')
        return 'protenix-v2'
    CFG['model_name'] = V2_FALLBACK
    print('\n' + '=' * 78)
    print(f'FALLBACK: protenix-v2 weights unavailable -> running {V2_FALLBACK}')
    print('(Protenix v1 architecture, training data up to 2025-06-30). Results are labelled with')
    print('this model in <run>/model_used.txt. Supply the v2 weights to use protenix-v2.')
    print('=' * 78 + '\n')
    return V2_FALLBACK

def run_pred(predict_json, run_dir, use_template, verbose=False):
    """protenix pred with Run-settings options. Samples per seed go through --sample (the pip
       CLI's option). If a build rejects an optional flag, retry without that flag only, and say
       loudly what was dropped - samples per seed is never dropped silently."""
    base = ['protenix', 'pred', '-i', predict_json, '-o', run_dir,
            '--seeds', CFG['seeds_csv'], '-n', CFG['model_name'],
            '--use_msa', str(CFG['use_msa']).lower(),
            '--use_template', str(use_template).lower(),
            '--use_rna_msa', str(CFG['use_rna_msa']).lower(),
            '--dtype', CFG['dtype']]
    core = ['--sample', str(CFG['n_sample'])]
    opts = []
    if CFG.get('use_tfg'):
        opts += ['--use_tfg_guidance', 'true']
    if CFG['opts']:
        # Fused CUDA kernels ship only up to sm_100; on Blackwell (sm_120) --enable_fusion fails
        # with "no kernel image" (reported async at a later op), so keep native ops there.
        if os.environ.get('TORCH_CUDA_ARCH_LIST', '') == '12.0':
            print('Blackwell (sm_120): --enable_fusion off (native ops); --enable_cache kept.')
            opts += ['--enable_cache', 'true']
        else:
            opts += ['--enable_fusion', 'true', '--enable_cache', 'true']
    os.makedirs(run_dir, exist_ok=True)
    with open(os.path.join(run_dir, 'model_used.txt'), 'w') as fh:
        fh.write(CFG['model_name'] + '\n')
    rejected = lambda o: ('o such option' in o) or ('unrecognized arguments' in o)
    rc, out = _stream(base + core + opts, verbose=verbose)
    if rc != 0 and rejected(out) and opts:
        print(f'\nThis build rejected an optional flag; retrying without {" ".join(opts[::2])}.')
        rc, out = _stream(base + core, verbose=verbose)
    if rc != 0 and rejected(out):
        print(f'\n*** WARNING: this Protenix build does not accept --sample; samples per seed '
              f'({CFG["n_sample"]}) is NOT applied - the build default is used. ***')
        rc, out = _stream(base, verbose=verbose)
    return rc, out

print('Helpers ready. Now run ONE input cell (4a-4d).')

## 3c. Protenix-v2 weights (downloads once, reused after)

Stages the `protenix-v2` checkpoint to `WORK/weights/protenix-v2.pt` (persists on Drive once 3b
has run) and into `/root/checkpoint/` where Protenix loads it. The first run downloads ~1.9 GB from
the HuggingFace mirror ([TMF001/protenix-v2-weights](https://huggingface.co/TMF001/protenix-v2-weights));
later runs find the saved file and skip the download. Optional — cell 6 / B2 also call this
automatically when `model_name = "protenix-v2"`. Other models don't need it.

If the weights can be neither found nor downloaded, inference falls back to `protenix_base_20250630_v1.0.0` (v1 architecture, newer training cutoff) and says so. A token for a gated mirror can be added as a Colab secret named `HF_TOKEN`.

In [ ]:
#@title 3c. Protenix-v2 weights (downloads once, reused after) { display-mode: "form" }
weights_dir = ""  #@param {type:"string"}
# Blank = default (WORK/weights, on Drive after step 3). Or a folder that holds protenix-v2.pt,
# or the path of the .pt file itself. If the mirror needs a token, add it as a Colab secret
# named HF_TOKEN (key icon in the left bar) before running this cell.
_ck = ensure_v2_weights(dest_dir=weights_dir.strip() or None)
if _ck:
    print('Protenix-v2 weights ready:', _ck)
else:
    print('\nprotenix-v2 is NOT usable yet - inference will fall back to '
          'protenix_base_20250630_v1.0.0 unless you provide the weights.')

## 5. Run settings

`protenix-v2` uses sensible defaults (N_cycle=10, N_step=200). With v2, ~5 seeds is plenty.
Templates and RNA-MSA require **local databases** (pdb_seqres / RNA DBs) and are off here.

In [ ]:
#@title Run settings { display-mode: "form" }
model_name = "protenix-v2"  #@param ["protenix-v2", "protenix_base_default_v1.0.0", "protenix_base_20250630_v1.0.0", "protenix_base_default_v0.5.0", "protenix_mini_default_v0.5.0"]
v2_weights_path = ""  #@param {type:"string"}
# ^ protenix-v2 only: folder holding protenix-v2.pt, or the .pt file. Blank = WORK/weights
#   (or the 3c path), else download. If none works -> protenix_base_20250630_v1.0.0.
seeds_mode = "list"  #@param ["list", "random_n"]
seeds_list = "101,102,103,104,105"  #@param {type:"string"}
random_n = 5  #@param {type:"slider", min:1, max:25, step:1}
samples_per_seed = 5  #@param {type:"slider", min:1, max:25, step:1}
use_msa = True  #@param {type:"boolean"}
msa_server_mode = "colabfold"  #@param ["colabfold", "protenix"]
use_tfg_guidance = False  #@param {type:"boolean"}
dtype = "bf16"  #@param ["bf16", "fp32"]
enable_speed_mem_opts = True  #@param {type:"boolean"}
use_templates = False  #@param {type:"boolean"}
use_rna_msa = False  #@param {type:"boolean"}
msa_timeout_min = 20  #@param {type:"integer"}
msa_retries = 2  #@param {type:"integer"}

import random
if seeds_mode == "list":
    SEEDS = [int(x) for x in seeds_list.split(',') if x.strip()]
else:
    SEEDS = random.sample(range(1, 100000), int(random_n))
SEEDS_CSV = ','.join(str(s) for s in SEEDS)

CFG = dict(model_name=model_name, v2_weights_path=v2_weights_path.strip(), seeds_csv=SEEDS_CSV, n_sample=int(samples_per_seed),
           use_msa=use_msa, msa_server_mode=msa_server_mode, use_tfg=use_tfg_guidance,
           dtype=dtype, opts=enable_speed_mem_opts, use_templates=use_templates,
           use_rna_msa=use_rna_msa, msa_timeout_min=int(msa_timeout_min),
           msa_retries=int(msa_retries))
print('Seeds:', SEEDS)
print(CFG)

## Batch input & run

Run many complexes in one go. A Protenix input file is a top-level **list** of job dicts, and
`protenix pred`/`msa` iterate over all of them — unique sequences are MSA-searched once and
reused across jobs. Build the list with `make_protenix_batch.py` (provided separately), then:
**B1** loads it, **B2** runs MSA + inference over the whole batch, and **B3–B6** produce a
per-job summary (ranked by ipTM), a master confidence table, PAE exports, and ipSAE
interface scores. Each job lands in `OUT_DIR/<job name>/`. Set options in **Run settings**
(cell 5) first.

In [ ]:
#@title B1. Batch input JSON (batch input list) { display-mode: "form" }
# A Protenix batch file is a top-level LIST of job dicts (each the same schema as a single-job
# input, one dict per complex). Point to a file on Drive, or leave blank to upload one.
# Build these with make_protenix_batch.py (provided separately).
batch_json_path = ""  #@param {type:"string"}

import os, json
if not batch_json_path:
    from google.colab import files
    up = files.upload()
    batch_json_path = os.path.join('/content', list(up.keys())[0])

BATCH_JSON = batch_json_path
batch = json.load(open(BATCH_JSON))
assert isinstance(batch, list) and batch, 'Batch JSON must be a non-empty LIST of job dicts.'
for b in batch:
    assert 'name' in b and 'sequences' in b, 'Each job needs a name and a sequences list.'
BATCH_NAMES = [b['name'] for b in batch]
assert len(BATCH_NAMES) == len(set(BATCH_NAMES)), 'Duplicate job names in batch - names must be unique.'
print(f'{len(BATCH_NAMES)} jobs loaded from {BATCH_JSON}:')
print(', '.join(BATCH_NAMES[:40]) + (' ...' if len(BATCH_NAMES) > 40 else ''))


### B2-pre. (Optional) Enable PAE capture - run BEFORE B2 to save dense PAE

In [ ]:
#@title B2-pre. (Optional) Enable PAE capture for the batch { display-mode: "form" }
# Run this BEFORE B2 if you want dense PAE matrices. It installs a tiny interpreter-startup patch
# that, as each job/seed is written during B2, also saves the expected PAE (Angstrom) as .npz next
# to the CIFs - no re-running. Sample order is forced to ORIGINAL so PAE/CIF/summary indices align
# (B3/B4 rank samples themselves, so this changes nothing downstream).
import os, textwrap

_patch_dir = '/content/_pae_patch'
os.makedirs(_patch_dir, exist_ok=True)
open(os.path.join(_patch_dir, 'sitecustomize.py'), 'w').write(textwrap.dedent('''
import os
if os.environ.get("PROTENIX_DUMP_PAE") == "1":
    try:
        import glob, numpy as np, torch
        import runner.inference as RI

        _orig_init = RI.InferenceRunner.init_dumper
        def _init_dumper(self, need_atom_confidence=False, sorted_by_ranking_score=True):
            # keep original sample order so pae_sample_i lines up with cif/summary sample_i
            return _orig_init(self, need_atom_confidence=need_atom_confidence,
                              sorted_by_ranking_score=False)
        RI.InferenceRunner.init_dumper = _init_dumper

        _orig_dump = RI.DataDumper.dump
        def _dump(self, *a, **k):
            out = _orig_dump(self, *a, **k)
            try:
                pred = k.get("pred_dict"); pdb_id = k.get("pdb_id"); seed = k.get("seed")
                if pred is not None and pred.get("pae") is not None:
                    lg = pred["pae"]
                    lg = lg if torch.is_tensor(lg) else torch.as_tensor(lg)
                    pr = torch.softmax(lg.float(), dim=-1)         # [S, N, N, bins]
                    nb = pr.shape[-1]
                    centers = (torch.arange(nb, dtype=torch.float32) + 0.5) * (32.0 / nb)
                    pae = (pr.cpu() * centers).sum(-1).numpy().astype("float16")   # [S, N, N] Angstrom
                    base = getattr(self, "base_dir", None) or getattr(self, "dump_dir", ".")
                    # Find THIS seed's output dir from the summary JSON just written by _orig_dump.
                    # (CIF names don't carry the seed; the seed lives in the folder name, e.g. seed_1.)
                    want = {str(seed), f"seed_{seed}", f"seed-{seed}"}
                    sjs = glob.glob(os.path.join(base, glob.escape(str(pdb_id)), "**",
                                    "*summary_confidence*.json"), recursive=True)
                    hit = [s for s in sjs if want & set(os.path.relpath(s, base).split(os.sep))] or sjs
                    if hit:
                        hit.sort(key=os.path.getmtime, reverse=True)
                        d = os.path.dirname(hit[0])
                        seed_dir = os.path.dirname(d) if os.path.basename(d) == "predictions" else d
                    else:
                        seed_dir = os.path.join(base, str(pdb_id), f"seed_{seed}")
                    pae_dir = os.path.join(seed_dir, "pae")     # -> <job>/seed_<seed>/pae
                    os.makedirs(pae_dir, exist_ok=True)
                    for i in range(pae.shape[0]):
                        np.savez_compressed(os.path.join(pae_dir,
                            f"{pdb_id}_{seed}_pae_sample_{i}.npz"), pae=pae[i])
                    print(f"[pae_dump] {pdb_id} seed {seed}: saved {pae.shape[0]} PAE matrices -> {pae_dir}")
            except Exception as e:
                print("[pae_dump] warning:", repr(e))
            return out
        RI.DataDumper.dump = _dump
        print("[pae_dump] active: PAE npz per sample (original order).")
    except Exception as e:
        print("[pae_dump] install failed:", repr(e))
'''))

os.environ['PROTENIX_DUMP_PAE'] = '1'
os.environ['PYTHONPATH'] = _patch_dir + os.pathsep + os.environ.get('PYTHONPATH', '')
print('PAE capture ENABLED. Run B2 as usual; <job>_<seed>_pae_sample_N.npz appears next to each CIF.')
print('Disable with:  os.environ["PROTENIX_DUMP_PAE"] = "0"')


### B2. Batch MSA + inference

In [ ]:
#@title B2. Batch MSA + inference { display-mode: "form" }
import os, datetime

assert 'CFG' in globals(), 'Run cell 5 (Run settings) first.'
assert 'BATCH_JSON' in globals(), 'Run B1 first.'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
VERBOSE = False   # True = show all of Protenix's INFO logs

# 0) Pick the model BEFORE the MSA step: protenix-v2 if its weights are available,
#    otherwise protenix_base_20250630_v1.0.0 (announced, and saved to model_used.txt).
MODEL_USED = resolve_model()

# 1) MSA: one ColabFold search per UNIQUE sequence, cached in MSA_DIR/_seqcache and reused
#    across jobs, batches and sessions. Searches still queued after msa_timeout_min are resubmitted.
stem = os.path.splitext(os.path.basename(BATCH_JSON))[0]
predict_json = BATCH_JSON
if CFG['use_msa']:
    if CFG['msa_server_mode'] == 'colabfold':
        predict_json = os.path.join(INPUT_DIR, f'{stem}-msa.json')
        if not msa_dedup(BATCH_JSON, predict_json, CFG['msa_timeout_min'], CFG['msa_retries']):
            raise SystemExit('MSA failed for the sequence(s) listed above. Re-run B2: finished '
                             'sequences are cached, so only the failed ones are searched again.')
    else:
        b_msa = os.path.join(MSA_DIR, stem); os.makedirs(b_msa, exist_ok=True)
        rc, _ = _stream(['protenix', 'msa', '--input', BATCH_JSON, '--out_dir', b_msa,
                         '--msa_server_mode', CFG['msa_server_mode']])
        if rc != 0:
            raise SystemExit('MSA step failed.')
        _upd = os.path.splitext(BATCH_JSON)[0] + '-update-msa.json'
        predict_json = _upd if os.path.exists(_upd) else BATCH_JSON

# 2) Predict all jobs into a dated run dir: RUN_DIR/<job>/seed_<n>/ (reruns never clash).
_stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
_bstem = stem.replace('-update-msa', '').replace('-msa', '')
RUN_DIR = os.path.join(OUT_DIR, f'run_{_stamp}_{_bstem}')
os.makedirs(RUN_DIR, exist_ok=True)
print('This batch ->', RUN_DIR)
use_template = CFG['use_templates'] and ('templatesPath' in open(predict_json).read())
rc, out = run_pred(predict_json, RUN_DIR, use_template, verbose=VERBOSE)
if rc != 0:
    raise SystemExit(f'Inference failed (exit {rc}).')
print('\nBatch done. Per-job outputs under', RUN_DIR)

### B3. Batch summary (best sample per job, ranked by ipTM)

In [ ]:
#@title B3. Batch summary (best sample per job, ranked by ipTM) { display-mode: "form" }
import os, json, numpy as np, pandas as pd
from IPython.display import display

_RUN = _run_dir()
files = _summary_jsons(_RUN)
names = list(globals().get('BATCH_NAMES') or sorted({_job_of(f, _RUN) for f in files}))
best = {}
for jf in files:
    job = _job_of(jf, _RUN)
    try:
        d = json.load(open(jf))
    except Exception:
        continue
    rs = d.get('ranking_score'); rs = float(rs) if rs is not None else -1.0
    if job not in best or rs > best[job]['ranking_score']:
        best[job] = dict(job=job, seed=_seed_of(jf), sample=_sample_of(jf), ranking_score=rs,
                         iptm=d.get('iptm'), ptm=d.get('ptm'), plddt=d.get('plddt'),
                         has_clash=d.get('has_clash'), **_pair_iptm(d),
                         cif=jf.replace('summary_confidence_', '').replace('.json', '.cif'))
rows = [best[n] for n in names if n in best]
assert rows, f'No outputs found under {_RUN}. Did B2 finish?'
df = pd.DataFrame(rows)
if df['n_chains'].max() < 3:     # two-chain jobs: the only chain pair IS ipTM, so no extra columns
    df = df.drop(columns=['chain_pair_iptm_max', 'chain_pair_iptm_min'])
df = df.sort_values('iptm', ascending=False, na_position='last').reset_index(drop=True)
_warn_na_seeds(df['seed'], 'B3')
pd.set_option('display.max_colwidth', 60)
display(df.drop(columns=['cif']))
df.to_csv(os.path.join(_RUN, 'batch_summary.csv'), index=False)
print('Saved', os.path.join(_RUN, 'batch_summary.csv'))
print('Best sample per job = highest ranking_score; jobs ranked by ipTM. For >=3-chain jobs '
      'chain_pair_iptm_max/min give the strongest/weakest chain pair.')
# auto-select the top hit so the single-model cells work immediately
JOB_SPEC = {'name': df.iloc[0]['job']}
print(f"\nJOB_SPEC set to top hit '{JOB_SPEC['name']}'. Set JOB_SPEC = {{'name': '<job>'}} for another.")
missing = [nm for nm in names if nm not in best]
if missing:
    print('\nNo output for:', ', '.join(missing), '(check the B2 log for per-job OOM/errors).')

### B4. Master confidence table - every model in the batch (CSV export)

In [ ]:
#@title B4. Master confidence table - every model in the batch { display-mode: "form" }
# Collects every summary_confidence JSON in the run into one table (one row per job x seed x
# sample) and writes batch_confidence_all_models.csv + batch_confidence_best_per_job.csv.
import os, json, numpy as np, pandas as pd
from IPython.display import display

_RUN = _run_dir()
rows = []
for jf in _summary_jsons(_RUN):
    try:
        d = json.load(open(jf))
    except Exception:
        continue
    row = {'job': _job_of(jf, _RUN), 'seed': _seed_of(jf), 'sample': _sample_of(jf)}
    for k, v in d.items():                       # all scalar confidences
        if isinstance(v, (int, float, bool)):
            row[k] = v
    row.update(_pair_iptm(d))                    # same chain-pair definition as B3/B7/viewer
    row['summary_path'] = jf
    rows.append(row)

df = pd.DataFrame(rows)
if len(df):
    if df['n_chains'].max() < 3:
        df = df.drop(columns=['chain_pair_iptm_max', 'chain_pair_iptm_min'])
    front = [c for c in ['job', 'seed', 'sample', 'ranking_score', 'plddt', 'ptm', 'iptm',
                         'n_chains', 'chain_pair_iptm_max', 'chain_pair_iptm_min',
                         'has_clash', 'gpde', 'disorder'] if c in df.columns]
    rest = [c for c in df.columns if c not in front and c != 'summary_path']
    _num = lambda s: s if s.name == 'job' else pd.to_numeric(s, errors='coerce')
    df = df[front + rest + ['summary_path']].sort_values(['job', 'seed', 'sample'], key=_num)
    df = df.reset_index(drop=True)
    _warn_na_seeds(df['seed'], 'B4')
    out_csv = os.path.join(_RUN, 'batch_confidence_all_models.csv')
    df.to_csv(out_csv, index=False)
    print(f'Wrote {out_csv}  |  {len(df)} models across {df["job"].nunique()} jobs')
    best = df.sort_values('iptm', ascending=False).groupby('job', as_index=False).first()
    best.to_csv(os.path.join(_RUN, 'batch_confidence_best_per_job.csv'), index=False)
    print('Also wrote batch_confidence_best_per_job.csv (highest-ipTM model per job).')
    display(df)
else:
    print('No summary_confidence JSONs found under', _RUN, '- run B2 first.')

### B5. PAE - best sample per seed (export + heatmap)

In [ ]:
#@title B5. PAE - best per seed: ChimeraX/ipSAE npz + heatmap { display-mode: "form" }
# For each (job, seed) picks the top-ranked sample and writes, into that seed's pae/ folder:
#   pae_<cifstem>.npz        (key 'pae', per-token; ligand-safe input for ChimeraX>=1.10 + ipSAE)
#   <job>_<seed>_pae_best.npz, _pae.json (ColabFold schema), _pae.png (heatmap w/ chain lines)
# Models are located via the summary JSONs (CIF derived from them), so naming/layout don't matter.
# Per-sample PAE files from B2-pre are kept unless prune_others is ticked.
import os, glob, json, re
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
try:
    import gemmi; _HAS_GEMMI = True
except Exception:
    _HAS_GEMMI = False

_RUN = _run_dir()
prune_others = False  #@param {type:"boolean"}
# ^ True deletes the per-sample <job>_<seed>_pae_sample_N.npz files saved by B2-pre,
#   keeping only the best sample per seed. Off by default: deletion cannot be undone.
round_dp = 2          #@param {type:"integer"}

def _cif_for(jf):
    cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
    if os.path.exists(cif):
        return cif
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    d = os.path.dirname(jf)
    cand = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
            else glob.glob(os.path.join(d, '*.cif')))
    return cand[0] if cand else None

def _seed_dir_of(cif):
    d = os.path.dirname(cif)
    return os.path.dirname(d) if os.path.basename(d) == 'predictions' else d

def _chain_boundaries(cif):
    if not (_HAS_GEMMI and cif and os.path.exists(cif)):
        return []
    try:
        st = gemmi.read_structure(cif); m = st[0]
        bnd, c = [], 0
        for ch in m:
            c += sum(1 for _ in ch); bnd.append(c)
        return bnd[:-1]
    except Exception:
        return []

def _heatmap(arr, cif, title, path):
    bnd = _chain_boundaries(cif)
    plt.figure(figsize=(5.2, 4.4))
    im = plt.imshow(arr, cmap='Greens_r', vmin=0, vmax=32)
    for b in bnd:
        plt.axhline(b - 0.5, color='k', lw=0.6); plt.axvline(b - 0.5, color='k', lw=0.6)
    plt.colorbar(im, label='PAE (Angstrom)')
    plt.title(title); plt.xlabel('Scored token'); plt.ylabel('Aligned token')
    plt.tight_layout(); plt.savefig(path, dpi=130); plt.close()

# best sample per (job, seed) via the summary JSONs
best = {}
for jf in _summary_jsons(_RUN):
    job = _job_of(jf, _RUN)
    seed = _seed_of(jf)
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    if not ms:
        continue
    try:
        rank = float(json.load(open(jf)).get('ranking_score') or 0)
    except Exception:
        rank = 0.0
    k = (job, seed)
    if k not in best or rank > best[k]['rank']:
        best[k] = {'rank': rank, 'jf': jf, 'N': ms.group(1)}

consolidated = os.path.join(_RUN, '_pae'); os.makedirs(consolidated, exist_ok=True)
n_npz = n_json = n_png = 0
for (job, seed), r in sorted(best.items()):
    cif = _cif_for(r['jf'])
    if not cif:
        continue
    seed_dir = _seed_dir_of(cif)
    pae_dir = os.path.join(seed_dir, 'pae')
    src = os.path.join(pae_dir, f'{job}_{seed}_pae_sample_{r["N"]}.npz')
    if not os.path.exists(src):
        cand = glob.glob(os.path.join(pae_dir, f'*_pae_sample_{r["N"]}.npz'))
        src = cand[0] if cand else None
    if not src or not os.path.exists(src):
        continue
    os.makedirs(pae_dir, exist_ok=True)
    arr = np.load(src)['pae'].astype('float32')
    cifstem = os.path.basename(cif)[:-4]

    np.savez_compressed(os.path.join(pae_dir, f'pae_{cifstem}.npz'), pae=arr.astype('float32'))   # ChimeraX/ipSAE
    np.savez_compressed(os.path.join(pae_dir, f'{job}_{seed}_pae_best.npz'), pae=arr.astype('float16'))
    json.dump([{"predicted_aligned_error": np.round(arr, round_dp).tolist(),
                "max_predicted_aligned_error": float(arr.max())}],
              open(os.path.join(pae_dir, f'{job}_{seed}_pae.json'), 'w'))
    _heatmap(arr, cif, f'{job}  seed {seed}', os.path.join(pae_dir, f'{job}_{seed}_pae.png'))
    np.savez_compressed(os.path.join(consolidated, f'{job}_seed{seed}_pae.npz'), pae=arr.astype('float16'))
    n_npz += 1; n_json += 1; n_png += 1

print(f'Best-per-seed: {n_npz} Boltz/ChimeraX npz + {n_json} JSON + {n_png} heatmaps, each in <job>/seed_<seed>/pae/.')
print('Consolidated npz copies ->', consolidated)

if prune_others:
    removed = 0
    for p in glob.glob(os.path.join(_RUN, '**', '*_pae_sample_*.npz'), recursive=True):
        if _in_archive(p, _RUN):
            continue
        try:
            os.remove(p); removed += 1
        except Exception:
            pass
    print(f'Pruned {removed} per-sample PAE files (kept best per seed).')

if best:
    (job, seed), r = max(best.items(), key=lambda kv: kv[1]['rank'])
    png = glob.glob(os.path.join(_RUN, job, '**', f'{job}_{seed}_pae.png'), recursive=True)
    if png:
        from IPython.display import Image, display
        print(f'Top model: {job} seed {seed} (rank {r["rank"]:.3f})')
        display(Image(filename=png[0]))
print('ChimeraX >=1.10 (ligand-safe):  open <cif>  then  open pae_<cifstem>.npz structure #1')
print('ChimeraX older / protein-only:  open <cif>  then  open <job>_<seed>_pae.json format pae')


### B6. ipSAE interface scores (Dunbrack, Boltz mode)

In [ ]:
#@title B6. ipSAE interface scores (Dunbrack, Boltz mode) { display-mode: "form" }
# Runs Dunbrack's ipsae.py on each best-per-seed model (B5's pae_<model>.npz + the CIF).
# Boltz mode also reads a per-token pLDDT file and the chain-pair ipTM; without them ipsae.py
# uses pLDDT = 0 (pDockQ/pDockQ2 stuck at 0.0183/0.0073) and ipTM = 0, so both are staged here.
# A value that cannot be computed is written as NA, never as a constant.
import os, glob, json, shutil, subprocess, sys
import numpy as np, pandas as pd

pae_cutoff  = 10   #@param {type:"number"}
dist_cutoff = 10   #@param {type:"number"}
_RUN = _run_dir()

IPSAE = '/content/ipsae.py'
if not (os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000):
    for url in ('https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py',
                'https://raw.githubusercontent.com/DunbrackLab/IPSAE/master/ipsae.py'):
        os.system(f'wget -q -O {IPSAE} "{url}"')
        if os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000:
            break
assert os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000, 'Could not download ipsae.py'

ps, ds = f'{int(pae_cutoff):02d}', f'{int(dist_cutoff):02d}'
WK = '/content/_ipsaework'            # neutral path: ipsae.py derives its input names by str.replace
rows, n_run, n_fail, n_noplddt, n_noiptm = [], 0, 0, 0, 0
npzs = [p for p in sorted(glob.glob(os.path.join(_RUN, '**', 'pae', 'pae_*.npz'), recursive=True))
        if not _in_archive(p, _RUN)]
for k, npz in enumerate(npzs):
    cifstem = os.path.basename(npz)[4:-4]                 # strip 'pae_' and '.npz'
    job, seed = _job_of(npz, _RUN), _seed_of(npz)
    seed_dir = os.path.dirname(os.path.dirname(npz))      # .../seed_<n>  (parent of pae/)
    cifs = glob.glob(os.path.join(seed_dir, '**', cifstem + '.cif'), recursive=True)
    if not cifs:
        continue
    cif, pae_dir = cifs[0], os.path.dirname(npz)
    summ = cif[:-4].replace('_sample_', '_summary_confidence_sample_') + '.json'
    wd = os.path.join(WK, f'm{k}'); shutil.rmtree(wd, ignore_errors=True)
    pae_p, cif_p, plddt_ok, iptm_ok = write_ipsae_inputs(wd, cif, np.load(npz)['pae'], summ)
    n_noplddt += not plddt_ok; n_noiptm += not iptm_ok
    r = subprocess.run([sys.executable, IPSAE, pae_p, cif_p, str(pae_cutoff), str(dist_cutoff)],
                       capture_output=True, text=True)
    n_run += 1
    outs = glob.glob(os.path.join(wd, f'model_{ps}_{ds}*'))
    if not any(o.endswith(f'_{ps}_{ds}.txt') for o in outs):
        n_fail += 1
        if n_fail <= 3:
            print(f'  no ipSAE output: {job} seed {seed}\n   {r.stderr.strip()[-300:]}')
        continue
    for o in outs:                                        # keep outputs next to the PAE, model-named
        shutil.move(o, os.path.join(pae_dir, cifstem + os.path.basename(o)[len('model'):]))
    shutil.rmtree(wd, ignore_errors=True)
    lines = [l.rstrip() for l in open(os.path.join(pae_dir, f'{cifstem}_{ps}_{ds}.txt')) if l.strip()]
    hdr = next((i for i, l in enumerate(lines) if l.split()[:2] == ['Chn1', 'Chn2']), None)
    if hdr is None:
        continue
    cols = lines[hdr].split()
    for l in lines[hdr + 1:]:
        parts = l.split()
        if len(parts) != len(cols):
            continue
        d = dict(zip(cols, parts))
        d.update(job=job, seed=seed, sample=_sample_of(cif), Model=cifstem)
        if not plddt_ok:
            d['pDockQ'] = d['pDockQ2'] = np.nan
        if not iptm_ok:
            d['ipTM_af'] = np.nan
        rows.append(d)

print(f'ipSAE run on {n_run} models ({n_fail} without output).')
if n_noplddt:
    print(f'  {n_noplddt} model(s): token count of CIF and PAE differ -> pDockQ/pDockQ2 = NA.')
if n_noiptm:
    print(f'  {n_noiptm} model(s): no chain_pair_iptm found -> ipTM_af = NA.')

if rows:
    df = pd.DataFrame(rows)
    keep_str = {'job', 'seed', 'sample', 'Chn1', 'Chn2', 'Type', 'Model'}
    for c in df.columns:
        if c not in keep_str:
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().any():
                df[c] = conv
    lead = [c for c in ['job', 'seed', 'sample', 'Chn1', 'Chn2'] if c in df.columns]
    df = df[lead + [c for c in df.columns if c not in lead]]
    _warn_na_seeds(df['seed'], 'B6')
    out_csv = os.path.join(_RUN, 'ipsae_all_chain_pairs.csv')
    df.to_csv(out_csv, index=False)
    print('Wrote', out_csv, '| rows:', len(df))
    if 'ipSAE' in df.columns:
        best = (df.sort_values('ipSAE', ascending=False).groupby('job', as_index=False).head(1)
                  .sort_values('ipSAE', ascending=False))
        best.to_csv(os.path.join(_RUN, 'ipsae_best_per_job.csv'), index=False)
        show = [c for c in ['job', 'seed', 'Chn1', 'Chn2', 'ipSAE', 'ipSAE_d0chn', 'ipTM_af',
                            'pDockQ', 'pDockQ2', 'LIS'] if c in best.columns]
        from IPython.display import display
        display(best[show].head(40))
    print('Note: in Boltz mode ipsae.py takes pLDDT per token (CA), so pDockQ can differ slightly '
          'from AF3-mode values, which use CB pLDDT.')
else:
    print('No ipSAE rows parsed - run B2-pre before B2 and B5 before this cell.')

### B7. Per-job means across seeds (pTM / ipTM / ipSAE consistency)

In [ ]:
#@title B7. Per-job means across seeds (consistency) { display-mode: "form" }
# Averages pTM / ipTM / ipSAE WITHIN each job across its seeds (best sample per seed by
# ranking_score), with std as the seed-to-seed spread.
import os, json
import numpy as np, pandas as pd
from IPython.display import display

_RUN = _run_dir()
rows = []
for jf in _summary_jsons(_RUN):
    try:
        d = json.load(open(jf))
    except Exception:
        continue
    rows.append(dict(job=_job_of(jf, _RUN), seed=_seed_of(jf), ranking_score=d.get('ranking_score'),
                     ptm=d.get('ptm'), iptm=d.get('iptm'), **_pair_iptm(d)))

df = pd.DataFrame(rows)
if df.empty:
    print('No summary JSONs found under', _RUN, '- run B2 first.')
else:
    _warn_na_seeds(df['seed'], 'B7')
    for c in ('ranking_score', 'ptm', 'iptm', 'chain_pair_iptm_max'):
        df[c] = pd.to_numeric(df[c], errors='coerce')
    best = (df.sort_values('ranking_score', ascending=False)
              .groupby(['job', 'seed'], as_index=False).head(1))
    g = best.groupby('job')
    agg = pd.DataFrame({
        'n_seeds':      g['seed'].nunique(),
        'iptm_mean':    g['iptm'].mean(),
        'iptm_std':     g['iptm'].std(),
        'ptm_mean':     g['ptm'].mean(),
        'ptm_std':      g['ptm'].std(),
        'ranking_mean': g['ranking_score'].mean(),
    })
    if df['n_chains'].max() >= 3:      # only meaningful (and not a copy of ipTM) with >=3 chains
        agg['chain_pair_iptm_max_mean'] = g['chain_pair_iptm_max'].mean()
        agg['chain_pair_iptm_max_std'] = g['chain_pair_iptm_max'].std()
    ip_csv = os.path.join(_RUN, 'ipsae_all_chain_pairs.csv')
    if os.path.exists(ip_csv):
        try:
            idf = pd.read_csv(ip_csv)
            idf['ipSAE'] = pd.to_numeric(idf.get('ipSAE'), errors='coerce')
            idf['seed'] = idf['seed'].astype(str)
            iseed = idf.groupby(['job', 'seed'], as_index=False)['ipSAE'].max()
            agg = agg.join(iseed.groupby('job')['ipSAE'].agg(ipsae_mean='mean', ipsae_std='std'))
        except Exception as e:
            print('ipSAE merge skipped:', e)
    else:
        print('(Run B6 first if you also want ipSAE means here.)')
    agg = agg.reset_index().sort_values('iptm_mean', ascending=False)
    out_csv = os.path.join(_RUN, 'batch_perjob_seed_means.csv')
    agg.round(4).to_csv(out_csv, index=False)
    print('Wrote', out_csv, '|', len(agg), 'jobs')
    display(agg.round(3))

### B8. Model viewer (MolView) + scores

In [ ]:
#@title Model viewer (MolView) + scores { display-mode: "form" }
# Pick any predicted model from the dropdowns; it renders in MolView (Mol*-based) colored by
# pLDDT, with its confidence scores shown. Works for a single job or a whole batch.
import os, glob, json, re, subprocess, sys
import numpy as np, pandas as pd

try:
    import molview as mv
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'molview'], check=False)
    import molview as mv
import ipywidgets as widgets
from IPython.display import display, clear_output
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

_RUN = _run_dir()

def _cif_for(jf):
    # same derivation cell 7 uses, with a glob fallback
    cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
    if os.path.exists(cif):
        return cif
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    d = os.path.dirname(jf)
    cand = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
            else glob.glob(os.path.join(d, '*.cif')))
    return cand[0] if cand else None

# optional ipSAE table (present only after the ipSAE cell has run)
_ipsae = None
_ip_csv = os.path.join(_RUN, 'ipsae_all_chain_pairs.csv')
if os.path.exists(_ip_csv):
    try:
        _ipsae = pd.read_csv(_ip_csv)
        _ipsae['seed'] = _ipsae['seed'].astype(str)
        _ipsae['ipSAE'] = pd.to_numeric(_ipsae.get('ipSAE'), errors='coerce')
    except Exception:
        _ipsae = None

# discover models the same way cell 7 / B-cells do: via the summary JSONs
recs = []
for jf in _summary_jsons(_RUN):
    try:
        d = json.load(open(jf))
    except Exception:
        continue
    cif = _cif_for(jf)
    if not cif:
        continue
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    recs.append(dict(job=_job_of(jf, _RUN),
                     seed=_seed_of(jf), sample=int(ms.group(1)) if ms else 0,
                     cif=cif, ranking=d.get('ranking_score', float('nan')),
                     iptm=d.get('iptm', float('nan')), ptm=d.get('ptm', float('nan')),
                     **_pair_iptm(d), plddt=d.get('plddt'),
                     has_clash=d.get('has_clash')))
mdf = pd.DataFrame(recs)

if mdf.empty:
    print('No models found under', _RUN)
    print('Looked for *summary_confidence*.json - has the inference cell finished and written here?')
else:
    mdf = mdf.sort_values('ranking', ascending=False).reset_index(drop=True)
    jobs = list(dict.fromkeys(mdf['job']))            # best-ranked job first
    job_dd   = widgets.Dropdown(options=jobs, description='Job:',
                                layout=widgets.Layout(width='70%'))
    model_dd = widgets.Dropdown(description='Model:', layout=widgets.Layout(width='70%'))
    color_dd = widgets.Dropdown(options=['plddt', 'chain', 'rainbow', 'secondary'],
                                value='plddt', description='Color:',
                                layout=widgets.Layout(width='40%'))
    out = widgets.Output()

    def _opts(job):
        sub = mdf[mdf['job'] == job]
        return [(f"seed {r['seed']} \u00b7 sample {r['sample']}  |  ipTM {r['iptm']:.2f}  "
                 f"pTM {r['ptm']:.2f}  rank {r['ranking']:.3f}", r['cif'])
                for _, r in sub.iterrows()]

    def _render(cif):
        with out:
            clear_output(wait=True)
            r = mdf[mdf['cif'] == cif].iloc[0]
            row = {'plddt': r['plddt'], 'pTM': r['ptm'], 'ipTM': r['iptm'],
                   'ranking_score': r['ranking'], 'has_clash': r['has_clash']}
            if r['n_chains'] >= 3:   # with 2 chains the only pair IS ipTM
                row['chain-pair ipTM max'] = r['chain_pair_iptm_max']
                row['chain-pair ipTM min'] = r['chain_pair_iptm_min']
            if _ipsae is not None:
                sel = _ipsae[(_ipsae['job'] == r['job']) & (_ipsae['seed'] == str(r['seed']))]
                if len(sel) and 'ipSAE' in sel.columns:
                    row['ipSAE (max pair)'] = float(sel['ipSAE'].max())
            print(f"{r['job']}  |  seed {r['seed']}  \u00b7  sample {r['sample']}")
            display(pd.DataFrame([row]).T.rename(columns={0: 'value'}))
            try:
                v = mv.view(width=720, height=520, panel=True)
                with open(cif) as f:
                    v.addModel(f.read())
                v.setColorMode(color_dd.value)
                v.show()
            except Exception as e:
                print('MolView render failed:', e)
                print('Open this file in ChimeraX instead:', cif)

    def _on_job(ch):
        if ch['name'] == 'value':
            model_dd.options = _opts(ch['new'])
            if model_dd.options:
                model_dd.value = model_dd.options[0][1]

    def _on_model(ch):
        if ch['name'] == 'value' and ch['new']:
            _render(ch['new'])

    def _on_color(ch):
        if ch['name'] == 'value' and model_dd.value:
            _render(model_dd.value)

    job_dd.observe(_on_job, names='value')
    model_dd.observe(_on_model, names='value')
    color_dd.observe(_on_color, names='value')

    display(widgets.HBox([job_dd]), widgets.HBox([model_dd, color_dd]), out)
    model_dd.options = _opts(jobs[0])
    if model_dd.options:
        model_dd.value = model_dd.options[0][1]
        _render(model_dd.value)


### B9. Interface contacts — per-job table (PAE substitute)

In [ ]:
#@title 9b. Interface contacts & residue-residue reliability (+ per-job table) { display-mode: "form" }
# Builds the inter-chain contact map for each job's best model, lists the closest residue pairs
# with their pLDDT, and reports how often each contact recurs across that job's samples/seeds.
# Exports one table per job (and a combined CSV); shows the map + table for the current top job.
import gemmi, numpy as np, glob, os, json, re
import pandas as pd
import matplotlib.pyplot as plt

_RUN = _run_dir()
CONTACT = 8.0          #@param {type:"number"}
max_rows_print = 30    #@param {type:"integer"}

def load_chains(cif):
    st = gemmi.read_structure(cif); m = st[0]
    out = {}
    for ch in m:
        rl = []
        for res in ch:
            coords = np.array([[a.pos.x, a.pos.y, a.pos.z] for a in res], dtype=float)
            if len(coords) == 0:
                continue
            ca = [[a.pos.x, a.pos.y, a.pos.z] for a in res if a.name == 'CA']
            rep = np.array(ca[0]) if ca else coords.mean(0)
            tab = gemmi.find_tabulated_residue(res.name)
            code = tab.one_letter_code.upper() if tab else 'X'
            rl.append({'num': res.seqid.num, 'code': code, 'rep': rep,
                       'plddt': float(np.mean([a.b_iso for a in res]))})
        if rl:
            out[ch.name] = rl
    return out

def _best_cif(job):
    best, br = None, -1e9
    for jf in glob.glob(os.path.join(_RUN, job, '**', '*summary_confidence*sample_*.json'), recursive=True):
        try:
            r = float(json.load(open(jf)).get('ranking_score') or 0)
        except Exception:
            r = 0.0
        if r > br:
            cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
            if not os.path.exists(cif):
                ms = re.search(r'sample_(\d+)', os.path.basename(jf)); d = os.path.dirname(jf)
                c = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
                     else glob.glob(os.path.join(d, '*.cif')))
                cif = c[0] if c else None
            if cif:
                br, best = r, cif
    return best

def contacts_table(job, cutoff=CONTACT):
    """Unique inter-chain residue pairs < cutoff for the job's best model, with recurrence
       across the job's samples. Returns (rows, D, A, B, cA, cB, n_used)."""
    cif = _best_cif(job)
    if not cif:
        return [], None, None, None, None, None, 0
    chains = load_chains(cif); names = list(chains)
    if len(names) < 2:
        return [], None, None, None, None, None, 0
    cA, cB = names[0], names[1]                     # first two chains; edit for another pair
    A, B = chains[cA], chains[cB]
    PA = np.array([r['rep'] for r in A]); PB = np.array([r['rep'] for r in B])
    D = np.linalg.norm(PA[:, None, :] - PB[None, :, :], axis=-1)
    freq = np.zeros_like(D); n_used = 0
    for cf in glob.glob(os.path.join(_RUN, job, '**', '*sample_*.cif'), recursive=True):
        ch = load_chains(cf)
        if cA in ch and cB in ch and len(ch[cA]) == len(A) and len(ch[cB]) == len(B):
            pa = np.array([r['rep'] for r in ch[cA]]); pb = np.array([r['rep'] for r in ch[cB]])
            freq += (np.linalg.norm(pa[:, None, :] - pb[None, :, :], axis=-1) < cutoff); n_used += 1
    freq /= max(n_used, 1)
    ii, jj = np.where(D < cutoff)
    rows, seen = [], set()
    for i, j in sorted(zip(ii, jj), key=lambda p: D[p]):
        key = (A[i]['num'], B[j]['num'])
        if key in seen:
            continue
        seen.add(key)
        rows.append(dict(job=job, chainA=cA, resA=A[i]['num'], codeA=A[i]['code'],
                         plddtA=round(A[i]['plddt'], 1), chainB=cB, resB=B[j]['num'],
                         codeB=B[j]['code'], plddtB=round(B[j]['plddt'], 1),
                         distance=round(float(D[i, j]), 2), recurrence=round(float(freq[i, j]), 3)))
    return rows, D, A, B, cA, cB, n_used

# ---- per-job export across every job in the run ----
jobs = sorted(d for d in os.listdir(_RUN)
              if os.path.isdir(os.path.join(_RUN, d)) and not d.startswith('_'))
all_rows = []
for job in jobs:
    rows = contacts_table(job)[0]
    if rows:
        all_rows += rows
        pd.DataFrame(rows).to_csv(os.path.join(_RUN, job, 'interface_contacts.csv'), index=False)

if all_rows:
    df = pd.DataFrame(all_rows)
    out_csv = os.path.join(_RUN, 'interface_contacts_all_jobs.csv')
    df.to_csv(out_csv, index=False)
    print(f'Wrote {out_csv}  |  {len(df)} contacts across {df["job"].nunique()} job(s).')
    print('Per-job copies: <job>/interface_contacts.csv')
else:
    df = None
    print('No inter-chain contacts found (monomers, or no >=2-chain jobs).')

# ---- interactive map + table for the current top job ----
_js = globals().get('JOB_SPEC') or {}
view_job = (_js.get('name') if _js.get('name') in jobs
            else (df.iloc[0]['job'] if df is not None else (jobs[0] if jobs else None)))
if view_job:
    rows, D, A, B, cA, cB, n_used = contacts_table(view_job)
    if D is not None:
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(D, cmap='viridis_r', vmin=0, vmax=20, aspect='auto')
        ax.set_xlabel(f'chain {cB} residue'); ax.set_ylabel(f'chain {cA} residue')
        ax.set_title(f'{view_job}: inter-chain distances | contacts < {CONTACT} A '
                     f'(recurrence over {n_used} models)')
        fig.colorbar(im, ax=ax, label='distance (Angstrom)'); plt.tight_layout(); plt.show()
        from IPython.display import display
        print(f'\nTop {min(max_rows_print, len(rows))} of {len(rows)} contacts for {view_job} '
              f'(chains {cA}-{cB}):\n')
        display(pd.DataFrame(rows).head(max_rows_print))
        print('\nHigh recurrence (~1.0) + high pLDDT on both partners = trustworthy specific contact.')


## 10. Zip & download all outputs

In [ ]:
#@title 10. Zip & download all outputs { display-mode: "form" }
import shutil, os
job = JOB_SPEC['name']
zip_base = os.path.join(WORK, job + '_protenix_v2')
_RUN = _run_dir()
shutil.make_archive(zip_base, 'zip', os.path.join(_RUN, job))
zp = zip_base + '.zip'
print('Zipped:', zp, f'({os.path.getsize(zp)/1e6:.1f} MB)')
try:
    from google.colab import files
    files.download(zp)
except Exception as e:
    print('Download manually from the Files panel:', zp, '|', e)

## 10b. Archive this run (dated, seed-labelled - prevents rerun overwrites)

In [ ]:
#@title 10b. Archive this run (date prefix + seed suffix, no overwrites) { display-mode: "form" }
# Copies the run into OUT_DIR/_archive/<STAMP>/ keeping the folder tree (<job>/seed_<n>/...), and
# renames each file <STAMP>_<name>_seed<n>.<ext>. Because the tree is kept, the results cells read
# an archive correctly: set RUN_DIR = '<OUT_DIR>/_archive/<STAMP>' and re-run B3/B4/B7.
import os, glob, shutil, datetime, zipfile

make_zip = True  #@param {type:"boolean"}

STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M')
_RUN = _run_dir()
if 'BATCH_NAMES' in globals() and BATCH_NAMES:
    jobs = list(BATCH_NAMES)
elif 'JOB_SPEC' in globals() and JOB_SPEC:
    jobs = [JOB_SPEC['name']]
else:
    jobs = sorted(d for d in os.listdir(_RUN) if os.path.isdir(os.path.join(_RUN, d)) and not d.startswith('_'))
if not jobs:
    raise SystemExit('Nothing to archive.')

archive_root = os.path.join(OUT_DIR, '_archive', STAMP)
n = 0
for nm in jobs:
    jdir = os.path.join(_RUN, nm)
    if not os.path.isdir(jdir):
        continue
    for f in glob.glob(os.path.join(jdir, '**', '*'), recursive=True):
        if not os.path.isfile(f):
            continue
        seed = _seed_of(f)
        stem, ext = os.path.splitext(os.path.basename(f))
        dst_dir = os.path.join(archive_root, os.path.relpath(os.path.dirname(f), _RUN))
        os.makedirs(dst_dir, exist_ok=True)
        tag = f'_seed{seed}' if seed != 'NA' else ''
        shutil.copy2(f, os.path.join(dst_dir, f'{STAMP}_{stem}{tag}{ext}'))
        n += 1
for f in glob.glob(os.path.join(_RUN, '*.csv')):          # run-level tables too
    os.makedirs(archive_root, exist_ok=True)
    shutil.copy2(f, os.path.join(archive_root, f'{STAMP}_{os.path.basename(f)}'))

print(f'Archived {n} files from {len(jobs)} job(s) -> {archive_root}')
print(f"Re-analyse later with:  RUN_DIR = '{archive_root}'")

if make_zip:
    zpath = archive_root + '.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(os.path.join(archive_root, '**', '*'), recursive=True):
            if os.path.isfile(f):
                z.write(f, os.path.relpath(f, archive_root))
    print(f'Zipped -> {zpath} ({os.path.getsize(zpath)/1e6:.1f} MB)')
    try:
        from google.colab import files
        files.download(zpath)
    except Exception as e:
        print('Download from the Files panel:', zpath, '|', e)